[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/10_remat_checkpoint.ipynb)

# 🟡 Medium: Gradient Checkpointing with jax.checkpoint

*JAX Fundamentals*
Apply a deep stack of blocks with **gradient checkpointing** (rematerialization).

Each block is
$$h \leftarrow \tanh(h W_i)$$

Given `params` (a list of `(D, D)` weight matrices) and `x` of shape `(B, D)`,
apply every block in order and return the final `(B, D)` activations — with each
block wrapped in `jax.checkpoint`.

### Rules
- Wrap **each block**, not the whole chain
- The output and gradients must be **numerically identical** to the unwrapped version
- Use `jax.checkpoint` (a.k.a. `jax.remat`)

### The tradeoff
Reverse-mode autodiff normally saves every intermediate activation on the
forward pass so the backward pass can use them. For an `L`-layer network that is
`O(L)` memory.

`jax.checkpoint` says: *don't save this block's internals — recompute them during
the backward pass.* Memory drops to `O(1)` per checkpointed block at the cost of
one extra forward evaluation. Checkpointing every layer takes peak memory from
`O(L)` to `O(1)` for about 1.3x the compute.

### Why it matters
This is exactly how large transformers are trained — one `checkpoint` per
transformer layer is standard practice, and it is often the difference between
a model fitting in HBM and not. Expect a follow-up question about where the
extra compute comes from, and about `policy=` for saving only the expensive ops
(like matmuls) while rematerializing the cheap elementwise ones.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def block(W, h):
    """One block: a matmul followed by a tanh."""
    return jnp.tanh(h @ W)


def deep_chain(params, x):
    """Apply every block in `params` to x, checkpointing each one.

    Args:
        params: list of (D, D) weight matrices
        x:      (B, D) input activations

    Returns:
        (B, D) final activations.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

keys = jax.random.split(jax.random.key(0), 4)
params = [jax.random.normal(k, (8, 8)) * 0.5 for k in keys]
x = jax.random.normal(jax.random.key(1), (2, 8))

out = deep_chain(params, x)
print("output shape:", out.shape)

# The remat2 primitive in the gradient's jaxpr is the proof it is checkpointed.
jaxpr = str(jax.make_jaxpr(lambda p, v: jnp.sum(deep_chain(p, v)))(params, x))
print("checkpointed:", "remat" in jaxpr)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("remat_checkpoint")

# hint("remat_checkpoint")      # stuck? nudge without the answer
# solution("remat_checkpoint")  # spoiler: the reference implementation